In [1]:
import optuna, pandas as pd
import tuning as tn

optuna.logging.set_verbosity(optuna.logging.WARNING)

df = tn.load_tuning_data()
feats = tn.load_feature_set("full", df)
print(df.shape, len(feats))   # expect 4510 rows, 101 features

s = tn.run_study("xgboost", horizon=1, df=df, features=feats, feature_set="full",
                 owner="smoke_test", task="reg", n_trials=2, storage=None, use_mlflow=False)
print(s.best_value)

(4510, 123) 101
15.365401379774813


In [2]:
OWNER = "abdelrahman_rafiq"
studies = {}
for h in (1, 2, 3):
    for fs in ("full", "pruned"):
        f = tn.load_feature_set(fs, df)
        studies[(h, fs)] = tn.run_study("xgboost", horizon=h, df=df, features=f,
                                        feature_set=fs, task="reg", owner=OWNER)

2026/10/25 22:19:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/25 22:19:24 INFO mlflow.store.db.utils: Updating database tables
2026/10/25 22:19:26 INFO mlflow.tracking.fluent: Experiment with name 'PM25_week4_tuning' does not exist. Creating a new experiment.


In [3]:
best_fs = {}
for h in (1, 2, 3):
    full = studies[(h, "full")].best_value
    pruned = studies[(h, "pruned")].best_value
    best_fs[h] = "pruned" if pruned <= full else "full"  
    print(f"t+{h}: full={full:.3f}  pruned={pruned:.3f}  -> {best_fs[h]}")

t+1: full=14.894  pruned=14.868  -> pruned
t+2: full=17.006  pruned=16.987  -> pruned
t+3: full=17.503  pruned=17.496  -> pruned


In [4]:
tn.setup_mlflow()

final, rows = {}, []
for h in (1, 2, 3):
    fs = best_fs[h]
    feats = tn.load_feature_set(fs, df)
    params = studies[(h, fs)].best_params
    res = tn.cv_evaluate("xgboost", params, df, h, feats, task="reg")
    tn.log_cv_result(res, params=params, features=feats, feature_set=fs,
                     owner=OWNER, run_type="final")
    rows.append(tn.result_row(res, feature_set=fs, label="xgb_tuned", params=params))
    final[h] = dict(res=res, params=params, fs=fs)

tuned = pd.DataFrame(rows)
tuned.to_csv(tn.ART_DIR / "results" / "task3_xgboost_tuned.csv", index=False)
tuned[["horizon", "feature_set", "cv_mean", "cv_std", "mae_mean", "r2_mean"]].round(3)

,horizon,feature_set,cv_mean,cv_std,mae_mean,r2_mean
0,1,pruned,14.868,5.107,11.163,0.345
1,2,pruned,16.987,6.390,12.772,0.175
2,3,pruned,17.496,6.434,13.271,0.126


In [5]:
import optuna.visualization as vis

for h in (1, 2, 3):
    s = studies[(h, best_fs[h])]

    fig1 = vis.plot_optimization_history(s)
    fig2 = vis.plot_param_importances(s)

    fig1.show()
    fig2.show()

    fig1.write_html(tn.ART_DIR / f"xgb_h{h}_history.html")
    fig2.write_html(tn.ART_DIR / f"xgb_h{h}_param_importance.html")

In [6]:
base = pd.read_csv(tn.ART_DIR / "results" / "task1_default_baseline_cv.csv")
base = base[(base.model == "xgboost") & (base.task == "reg")]

b = base[["horizon", "feature_set", "cv_mean", "cv_std"]].rename(
    columns={"cv_mean": "default_rmse", "cv_std": "default_std"})
t = tuned[["horizon", "feature_set", "cv_mean", "cv_std"]].rename(
    columns={"cv_mean": "tuned_rmse", "cv_std": "tuned_std"})

compare = t.merge(b, on=["horizon", "feature_set"]).set_index("horizon")
compare["improvement_%"] = (1 - compare.tuned_rmse / compare.default_rmse) * 100
compare[["feature_set", "default_rmse", "default_std", "tuned_rmse", "tuned_std", "improvement_%"]].round(3)

,feature_set,default_rmse,default_std,tuned_rmse,tuned_std,improvement_%
horizon,,,,,,
1,pruned,15.258,5.352,14.868,5.107,2.551
2,pruned,17.563,6.566,16.987,6.390,3.280
3,pruned,17.901,6.679,17.496,6.434,2.263


In [9]:
import json

with open("task2_best_lightgbm_params.json") as f:
    raw = json.load(f)

lgbm_params = {int(k[2:]): v["best_params"] for k, v in raw.items()}        
lgbm_fs     = {int(k[2:]): v["chosen_feature_set"] for k, v in raw.items()}
lgbm_cv     = {int(k[2:]): v["tuned_cv_rmse"] for k, v in raw.items()}      

h2h = []
for h in (1, 2, 3):
    # subsample_freq=1 is fixed inside the LightGBM space, so it is NOT in best_params: add it back
    p = {**lgbm_params[h], "subsample_freq": 1}
    r_l = tn.cv_evaluate("lightgbm", p, df, h, tn.load_feature_set(lgbm_fs[h], df), task="reg")
    x = final[h]["res"].fold_metrics["rmse"].to_numpy()
    l = r_l.fold_metrics["rmse"].to_numpy()
    wins = int((x < l).sum())
    mean_x, mean_l = x.mean(), l.mean()
    verdict = ("XGBoost" if wins >= 4 and mean_x < mean_l
            else "LightGBM" if wins <= 1 and mean_l < mean_x
            else "tie")
    h2h.append(dict(horizon=h, xgb_rmse=x.mean(), lgbm_rmse=l.mean(), xgb_fold_wins=f"{wins}/5",
                    verdict=verdict))
    print(f"t+{h}: my rerun of LightGBM = {r_l.score:.3f} | Abdullah reported = {lgbm_cv[h]:.3f}")

pd.DataFrame(h2h).round(3)

t+1: my rerun of LightGBM = 14.926 | Abdullah reported = 14.926
t+2: my rerun of LightGBM = 16.909 | Abdullah reported = 16.909
t+3: my rerun of LightGBM = 17.533 | Abdullah reported = 17.533


,horizon,xgb_rmse,lgbm_rmse,xgb_fold_wins,verdict
0,1,14.868,14.926,2/5,tie
1,2,16.987,16.909,3/5,tie
2,3,17.496,17.533,1/5,tie


In [10]:
import json, optuna

summary = {}
for h in (1, 2, 3):
    fs = best_fs[h]
    s = studies[(h, fs)]
    summary[f"t+{h}"] = dict(
        chosen_feature_set=fs,
        full_best_rmse=studies[(h, "full")].best_value,
        pruned_best_rmse=studies[(h, "pruned")].best_value,
        best_params=s.best_params,
        n_complete=sum(t.state.name == "COMPLETE" for t in s.trials),
        n_pruned=sum(t.state.name == "PRUNED" for t in s.trials),
        param_importance=optuna.importance.get_param_importances(s),
        xgb_fold_rmse=[float(v) for v in final[h]["res"].fold_metrics["rmse"]],
    )
json.dump(summary, open(tn.ART_DIR / "task3_summary.json", "w"), indent=2)

compare.to_csv(tn.ART_DIR / "results" / "task3_vs_default.csv")
pd.DataFrame(h2h).to_csv(tn.ART_DIR / "results" / "task3_vs_lightgbm.csv", index=False)